# Persistent Homology & Cocycles

This notebook will demonstrate the clean usage of `TopoVNL` based on the `MIMIC-MJX` package.

In [1]:
import os
os.chdir('..')

# Step 1: Imports

In [3]:
%env MUJOCO_GL=egl
%env PYOPENGL_PLATFORM=egl
%matplotlib inline
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import multiprocessing as mp
import functools
import matplotlib.pyplot as plt
import umap
import h5py

from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from vnl_tda.utils.load_utils import (
    load_from_h5py,
    get_aggregate_data,
)

from vnl_tda.utils.rendering import (
    render_from_saved_rollout,
    display_video,
)

from vnl_tda.pipelines.gaitcycle.gait_intention import (
    sampled_umap_with_gait
)

from vnl_tda.pipelines.tda.homology.taken_embedding import (
    compute_tda_per_clip_with_delayed_embedding,
    visualize_wasserstein_distance_matrix,
    cluster_and_plot_wasserstein_matrix,
    delayed_embedding_tda_then_projection,
)

from vnl_tda.pipelines.tda.skeletonization.skeletonization import (
    extract_generative_loops_cocycles_based,
    extract_generative_loops_cocycles_power_based,
)

rodent_rollout_dir = "/root/vast/kaiwen/TopoVNL/data/rodent_gaussian"
num_clips = 500

env: MUJOCO_GL=egl
env: PYOPENGL_PLATFORM=egl


/root/miniforge3/envs/track-mjx/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Check the keys of teh folder to ensure we have everything using the below:

In [4]:
with h5py.File(rodent_rollout_dir + "/clip_0.h5", "r") as f:
    print(list(f.keys()))

['ctrl', 'intention', 'obs', 'qpos', 'reward', 'sensors']


---

# Step 2: Loading & preprocessing

We can visualize the gait cycle of each of the virtuak rodent, just like activation, we can load in the  `joint forces` and `intention` for each first.

In [5]:
work_rodent = functools.partial(get_aggregate_data, "/sensors", [], path=rodent_rollout_dir)

with mp.Pool(processes=mp.cpu_count()) as pool:
    jf_rodent = list(tqdm(pool.imap(work_rodent, range(num_clips)), total=num_clips))

jf_rodent = np.vstack(jf_rodent)
jf_rodent = jf_rodent.reshape([num_clips, 499, 16])
jf_rodent = jf_rodent[:,:,9:13]

100%|██████████| 200/200 [00:00<00:00, 1718.98it/s]


`Note` that we will be using teh 60 dimension intentionchecpoiint for all TDA as they seem to be more mature in forming their representation of gait cycle and the 16 dimension one for gait analysis as we have more sensors there.

In [6]:
work_rodent = functools.partial(get_aggregate_data, "/intention", [], path=rodent_rollout_dir)

with mp.Pool(processes=mp.cpu_count()) as pool:
    int_rodent = list(tqdm(pool.imap(work_rodent, range(num_clips)), total=num_clips))

int_rodent = np.vstack(int_rodent)

100%|██████████| 200/200 [00:00<00:00, 1539.65it/s]


Kicking out bad performance clips identified by topological features

---

# Step 3: Intention manifold & gait groups

In [7]:
int_rodent = int_rodent.reshape(-1, 16)
window_size = 61
results = sampled_umap_with_gait(int_rodent, jf_rodent,
                                 window_size=window_size,
                                 stride=10, n_samples=10000
                                 )

---

# Step 4: Taken's delayed embeddings

## Step 4.1: Wasserstein distance on persistent signitures

This is a TDA conducted with more respect of the original time series nature of the data set:

In [20]:
tda_results_per_clip = compute_tda_per_clip_with_delayed_embedding(
    intentions=int_rodent,
    m=5,
    frames_per_clip=500,
    max_dim=2,
    max_points=500
)

In [3]:
np.savez_compressed("preprocessed_data/takens_embedding_tda_rodent.npz", tda_results=tda_results_per_clip)

For purpose of doing anlysis, we will be using an pre-computed tda dataset with takens embedding

In [8]:
# file_path = "data/preprocessed_data/takens_embedding_tda_rodent.npz"
# size_bytes = os.path.getsize(file_path)
# size_mb = size_bytes / (1024 * 1024)
# print(f"File size: {size_mb:.2f} MB")
# loaded = np.load(file_path, allow_pickle=True)
# tda_results_per_clip = loaded['tda_results'].tolist()

In [9]:
def get_all_keys(d, parent_key=''):
    keys = []
    for k, v in d.items():
        full_key = f"{parent_key}.{k}" if parent_key else k
        keys.append(full_key)
        if isinstance(v, dict):
            keys.extend(get_all_keys(v, full_key))
    return keys

print(get_all_keys(tda_results_per_clip[0]))

Notice that this is the `2d Wasserstein distance` matrix between two persistent diagram, so it is looking at the partial matching between two sets of points, which is fundamentally different comparing the `1d Wasserstein distance` that we use later.

In [10]:
wass_matrix = visualize_wasserstein_distance_matrix(tda_results_per_clip, max_clips=300, diag_type='H1', skip_clips=[37, 152, 163, 274, 281],)

We cna easily find out once that exert bad controls, the intention space for them is not really structured at all.

In [42]:
with h5py.File(
    rodent_rollout_dir + f"/clip_37.h5", "r"
) as h5file:
    rollout_rodent  = load_from_h5py(h5file)
        
frames = render_from_saved_rollout(rollout_rodent)
display_video(frames, framerate=50)

Then we can use **agglomerative (bottom-up) hierarchical clustering** to:
* Start with each clip as its own cluster.
* Merge the two closest clusters based on the average Wasserstein distance.
* Repeat merging until all clips are in a single cluster tree.

In [20]:
clip_clusters, linkage_matrix = cluster_and_plot_wasserstein_matrix(wass_matrix, n_clusters=4)

We can also conduct dimenisonality reduction to visualize the manifold of this manifold space!

In [ ]:
delayed_embedding_tda_then_projection(tda_results_per_clip, reduction='pca', n_components=3, n_clusters=2)
delayed_embedding_tda_then_projection(tda_results_per_clip, reduction='tsne', n_components=3,n_clusters=2)
delayed_embedding_tda_then_projection(tda_results_per_clip, reduction='umap', n_components=3, n_clusters=2)

In [ ]:
rows = []
for r in tda_results_per_clip:
    d  = r["diagrams"]
    h1 = d.get("H1", [])
    h2 = d.get("H2", [])
    h1s, h2s = d.get("H1_stats", {}), d.get("H2_stats", {})
    rows.append({
        "H1 Count": len(h1),
        "H1 Persistence Mean": h1s.get("mean",   0.0),
        "H1 Persistence Median": h1s.get("median", 0.0),
        "H1 Persistence Max": h1s.get("max",    0.0),
        "H2 Count": len(h2),
        "H2 Persistence Mean": h2s.get("mean",   0.0),
        "H2 Persistence Median": h2s.get("median", 0.0),
        "H2 Persistence Max": h2s.get("max",    0.0)
    })
    
df = pd.DataFrame(rows)
df.describe()

## Step 4.2: Background projection

Reload intention to use the latest AR intention:

In [4]:
rodent_rollout_dir = "/root/vast/kaiwen/TopoVNL/data/rodent_ar"
work_rodent = functools.partial(get_aggregate_data, "/intention", [], path=rodent_rollout_dir)

with mp.Pool(processes=mp.cpu_count()) as pool:
    int_rodent = list(tqdm(pool.imap(work_rodent, range(num_clips)), total=num_clips))

int_rodent = np.vstack(int_rodent)

100%|██████████| 500/500 [00:00<00:00, 5811.43it/s]


Firstm let's create a 3D background embedding from different dimensionality reduction projections (PCA, TSNE, UMAP) for visualization of the cycles that got identified
- For `window_size`, use 20 for rodent, for `stride`, use 10 for both body models

In [12]:
from vnl_tda.pipelines.gaitcycle.utils import (
    create_time_windows
)

int_rodent = int_rodent.reshape([num_clips, 499, 16])

window_size = 20
stride = 10
pts_per_clip = []
centers_per_clip = []
for clip in int_rodent:
    windows, centers = create_time_windows(
        clip,
        windows_size=window_size,
        stride=stride
    )
    pts_per_clip.append(windows)
    centers_per_clip.append(centers)

all_windows = np.vstack(pts_per_clip)
print(np.array(pts_per_clip).shape, all_windows.shape)

(500, 48, 320) (24000, 320)


In [13]:
embeds3d = []
emb_pca3d = PCA(n_components=3, random_state=42).fit_transform(all_windows)
embeds3d.append((emb_pca3d, "PCA 3D"))

emb_tsne3d = TSNE(n_components=3, perplexity=30, init="pca", random_state=42).fit_transform(all_windows)
embeds3d.append((emb_tsne3d, "t-SNE 3D"))

emb_umap3d = umap.UMAP(n_components=3, n_neighbors=15, random_state=42).fit_transform(all_windows)
embeds3d.append((emb_umap3d, "UMAP 3D"))

/root/miniforge3/envs/track-mjx/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


# Step 5: Traditional & DTM filtration Cocycle

Can we talk about the `complexity` or `stability` of the intention space? A large number of loop points usually means:

1. Many windows satisfy the `cocycle edges`  → the loop spans a long temporal stretch of the clip, indicating that the system stayed on, or repeatedly returned to, a coherent cyclic manifold for an extended portion of the clip
    - Notice that this doesn't neccessaily mean gait, but rather discusses about the stability of the intentions).
    - From graph we may tell when the disturbance in the intention space happens.

2. `TSP` re‑ordering yields hundreds of vertices, meaning that the points are numerous enough to form a smooth closed path after ordering. We can visualise a nearly continuous orbit in UMAP/PCA.

We should not be looking just at this as this is the clip labeling for new `H5` file data, does not work for the old ones.

In [46]:
loops_cocycle, loops_idx_cocycle, all_pts_cocycle = extract_generative_loops_cocycles_based(
    int_rodent,
    m=5,
    max_clips=1
)

In [47]:
loops_cocycle_knn, loops_idx_cocycle_knn, all_pts_cocycle_knn = extract_generative_loops_cocycles_power_based(
    int_rodent,
    m=5,
    max_clips=1,
    k=5,
)

We can also use the persistent method from previous cocycle to try to find these features --> the most persistent hole willbe corresponding to what umap have found

In [48]:
offset = 0
for cid_indicator, (pts, loop) in enumerate(zip(all_pts_cocycle_knn, loops_idx_cocycle_knn)):
    n = len(pts)
    if n == 0:
        offset += n
        continue

    loop_idx = np.array(loop, dtype=int)
    cycle_global = offset + loop_idx
    clip_global = np.arange(offset, offset + n)

    fig = plt.figure(figsize=(30, 8), dpi=150)
    for i, (emb, title) in enumerate(embeds3d, start=1):
        ax = fig.add_subplot(1, len(embeds3d), i, projection='3d')
        
        ax.scatter(emb[:,0], emb[:,1], emb[:,2],
           s=2, alpha=0.005, color="lightgrey")
        
        ax.scatter(emb[clip_global, 0],
                   emb[clip_global, 1],
                   emb[clip_global, 2],
                   s=8, alpha=0.3)
        
        ax.scatter(emb[cycle_global, 0],
                   emb[cycle_global, 1],
                   emb[cycle_global, 2],
                   s=20, color="tab:blue")
        
        loop_coords = emb[cycle_global]
        
        ax.set_title(title)
        
        ax.set_xlabel("Dim 1"); ax.set_ylabel("Dim 2"); ax.set_zlabel("Dim 3")
        ax.axis("off")
        
    fig.suptitle(f"Clip {cid_indicator} Loop in Global 3D Embeddings", fontsize=14)
    plt.show()
    offset += n

---